# Lab 02: Gradient-Based Optimization & Momentum Dynamics

[Back to Session Overview](../README.md) · [Module 02 Index](../../README.md)

---

## Overview

Optimization is the computational engine of machine learning. Once an objective loss function $\mathcal{L}(\theta)$ is formulated, training searches for the parameter vector $\theta^*$ that minimizes the loss.

In this notebook, we analyze and implement:
1. **The Direction of Steepest Descent:** Why stepping opposite to the gradient $\nabla f(x)$ minimizes a function.
2. **Step-by-Step Analytical Trace on $f(x) = x^2$:** Recreating and verifying the exact numerical steps from the lecture slides ($x_0 = 70.0 \to 56.0 \to 44.8 \to 35.84 \to 28.672$).
3. **The Role of the Learning Rate $\eta$:** Exploring under-stepping, exact single-step convergence, oscillatory behavior, and numerical divergence.
4. **Polyak's Heavy-Ball Momentum:** Introducing physical velocity to accelerate convergence through ravines and dampen high-frequency oscillations.
5. **2D Optimization on Anisotropic Quadratic Surfaces:** Visualizing optimization paths on ill-conditioned ravines.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---

## 1. Mathematical Principles of Gradient Descent

For a scalar objective function $f(x)$, the derivative $f'(x) = \frac{df}{dx}$ gives the instantaneous rate of increase of $f$:
- If $f'(x) > 0$: The function is increasing to the right; to decrease $f(x)$, we must move **left** (decrement $x$).
- If $f'(x) < 0$: The function is decreasing to the right; to decrease $f(x)$, we must move **right** (increment $x$).

This establishes the universal **Gradient Descent update rule**:

$$x_{t+1} = x_t - \eta f'(x_t)$$

where $\eta > 0$ denotes the learning rate (step size).


---

## 2. Recreating the Lecture Trace: Minimizing $f(x) = x^2$

Consider minimizing $f(x) = x^2$ subject to $-100 \le x \le 100$.
- Analytical derivative: $f'(x) = 2x$
- Initialization: $x_0 = 70.0$
- Learning rate: $\eta = 0.1$

Let us compute the iterations step-by-step:
- **Iteration 0:**
  $$f'(x_0) = 2(70.0) = 140.0$$
  $$x_1 = x_0 - \eta f'(x_0) = 70.0 - 0.1(140.0) = 56.0$$

- **Iteration 1:**
  $$f'(x_1) = 2(56.0) = 112.0$$
  $$x_2 = x_1 - \eta f'(x_1) = 56.0 - 0.1(112.0) = 44.8$$

- **Iteration 2:**
  $$f'(x_2) = 2(44.8) = 89.6$$
  $$x_3 = x_2 - \eta f'(x_2) = 44.8 - 0.1(89.6) = 35.84$$

- **Iteration 3:**
  $$f'(x_3) = 2(35.84) = 71.68$$
  $$x_4 = x_3 - \eta f'(x_3) = 35.84 - 0.1(71.68) = 28.672$$

Notice the geometric progression: $x_{t+1} = x_t - 0.2 x_t = 0.8 x_t = (0.8)^t x_0$.


In [ ]:
def gradient_descent_trace(x0, eta, num_steps):
    """Execute Gradient Descent on f(x) = x^2 and record every intermediate state."""
    history = []
    x = float(x0)
    for t in range(num_steps):
        grad = 2.0 * x
        f_val = x**2
        history.append((t, x, grad, f_val))
        x = x - eta * grad
    return history


trace = gradient_descent_trace(x0=70.0, eta=0.1, num_steps=5)

header_grad = "f '(x_t)"
print(f"{'Step t':<8} | {'x_t':<12} | {header_grad:<12} | {'f(x_t)':<12}")
print("-" * 52)
for t, x_val, grad_val, f_val in trace:
    print(f"{t:<8} | {x_val:<12.4f} | {grad_val:<12.4f} | {f_val:<12.4f}")

# Verify exact match against lecture slide numbers
assert np.isclose(trace[0][1], 70.0)
assert np.isclose(trace[0][2], 140.0)
assert np.isclose(trace[1][1], 56.0)
assert np.isclose(trace[1][2], 112.0)
assert np.isclose(trace[2][1], 44.8)
assert np.isclose(trace[2][2], 89.6)
assert np.isclose(trace[3][1], 35.84)
assert np.isclose(trace[3][2], 71.68)
assert np.isclose(trace[4][1], 28.672)
print("\nAll lecture trace values match exactly to floating-point precision!")

---

## 3. Impact of Learning Rate $\eta$ on Convergence Dynamics

For the quadratic $f(x) = a x^2$, the recurrence is:

$$x_t = (1 - 2 a \eta) x_{t-1} = (1 - 2 a \eta)^t x_0$$

The contraction factor is $\rho = 1 - 2 a \eta$:
1. $0 < \eta < \frac{1}{2a}$: Monotonic geometric decay ($0 < \rho < 1$).
2. $\eta = \frac{1}{2a}$: Exact convergence to minimum in a **single step** ($ho = 0$). For $a = 1$, $\eta = 0.5$.
3. $\frac{1}{2a} < \eta < \frac{1}{a}$: Damped oscillations about the minimum ($-1 < \rho < 0$).
4. $\eta = \frac{1}{a}$: Undamped infinite oscillations between $\pm x_0$ ($\rho = -1$).
5. $\eta > \frac{1}{a}$: Numerical explosion and divergence ($|\rho| > 1$).


In [ ]:
etas = [0.05, 0.1, 0.5, 0.8, 1.0, 1.05]
num_steps = 15

plt.figure(figsize=(12, 7))
for eta in etas:
    trace = gradient_descent_trace(x0=70.0, eta=eta, num_steps=num_steps)
    xs = [pt[1] for pt in trace]
    plt.plot(range(num_steps), xs, marker="o", label=f"eta = {eta}")

plt.axhline(0, color="black", linestyle="--", alpha=0.6)
plt.title(
    "Convergence Dynamics of Gradient Descent on f(x) = x^2 under Varying Learning Rates",
    fontsize=13,
)
plt.xlabel("Iteration t", fontsize=11)
plt.ylabel("Parameter Value x_t", fontsize=11)
plt.ylim(-100, 120)
plt.legend(loc="upper right")
plt.show()

# Assertions verifying theoretical convergence modes
assert np.isclose(gradient_descent_trace(70.0, eta=0.5, num_steps=2)[1][1], 0.0), (
    "eta=0.5 must hit 0 in 1 step"
)
osc_trace = gradient_descent_trace(70.0, eta=1.0, num_steps=3)
assert np.isclose(osc_trace[1][1], -70.0) and np.isclose(osc_trace[2][1], 70.0), (
    "eta=1.0 must oscillate indefinitely"
)

---

## 4. Momentum-Accelerated Optimization (Polyak Heavy-Ball)

Standard Gradient Descent struggles in ravines where the surface curves much more steeply in one dimension than another. The parameter oscillates wildly across the steep direction while creeping slowly along the flat valley.

**Polyak's Heavy-Ball Momentum** introduces a velocity state vector $v_t$:

$$v_t = \gamma v_{t-1} + \eta \nabla f(x_{t-1})$$
$$x_t = x_{t-1} - v_t$$

where $\gamma \in [0, 1)$ is the momentum decay factor (typically $0.9$).

By accumulating velocity along consistent directions, momentum accelerates effective step size by $\frac{1}{1 - \gamma}$ ($10\times$ when $\gamma = 0.9$) while canceling out alternating transverse oscillations.


In [ ]:
def momentum_descent_2d(grad_fn, x0, y0, eta=0.1, gamma=0.9, num_steps=50):
    """2D Momentum Optimizer."""
    x, y = float(x0), float(y0)
    vx, vy = 0.0, 0.0
    traj = [(x, y)]
    for _ in range(num_steps):
        gx, gy = grad_fn(x, y)
        vx = gamma * vx + eta * gx
        vy = gamma * vy + eta * gy
        x = x - vx
        y = y - vy
        traj.append((x, y))
    return np.array(traj)


def standard_gd_2d(grad_fn, x0, y0, eta=0.1, num_steps=50):
    """2D Standard Gradient Descent."""
    x, y = float(x0), float(y0)
    traj = [(x, y)]
    for _ in range(num_steps):
        gx, gy = grad_fn(x, y)
        x = x - eta * gx
        y = y - eta * gy
        traj.append((x, y))
    return np.array(traj)


# Test on anisotropic ravine: f(x, y) = 0.05 * x^2 + 1.0 * y^2
# Curvature ratio = 1.0 / 0.05 = 20:1 (condition number = 20)
def grad_ravine(x, y):
    return (0.1 * x, 2.0 * y)


traj_gd = standard_gd_2d(grad_ravine, x0=10.0, y0=5.0, eta=0.8, num_steps=40)
traj_mom = momentum_descent_2d(grad_ravine, x0=10.0, y0=5.0, eta=0.1, gamma=0.85, num_steps=40)

# Plot comparison on contour map
x_grid = np.linspace(-2, 12, 200)
y_grid = np.linspace(-6, 6, 200)
X, Y = np.meshgrid(x_grid, y_grid)
Z = 0.05 * X**2 + 1.0 * Y**2

plt.figure(figsize=(12, 6))
plt.contour(X, Y, Z, levels=30, cmap="viridis", alpha=0.5)
plt.plot(
    traj_gd[:, 0],
    traj_gd[:, 1],
    "ro-",
    label="Standard GD (eta=0.8: severe oscillation)",
    lw=1.5,
    ms=4,
)
plt.plot(
    traj_mom[:, 0],
    traj_mom[:, 1],
    "b*-",
    label="Momentum (eta=0.1, gamma=0.85: smooth descent)",
    lw=2,
    ms=5,
)
plt.plot(0, 0, "gX", ms=12, label="Global Minimum (0, 0)")
plt.title(
    "Optimization Trajectories on an Anisotropic Quadratic Ravine (Condition Number = 20)",
    fontsize=13,
)
plt.xlabel("x (flat direction)", fontsize=11)
plt.ylabel("y (steep direction)", fontsize=11)
plt.legend()
plt.show()